# AgriShield AI: Unified Plant & Medicinal Leaf Disease Classifier

This notebook downloads, merges, and trains a PyTorch image classification model on three different Kaggle datasets:
1. **`vipoooool/new-plant-diseases-dataset`** (Standard agricultural crops, 38 classes)
2. **`jocelyndumlao/medicinal-leaf-health-classification`** (Medicinal plant leaves, background-removed, 7 species, multiple health conditions)
3. **`rashikrahmanpritom/plant-disease-recognition-dataset`** (Generic leaf diseases: Rust, Powdery Mildew, Healthy)

### Workflow:
1. Download datasets using `kagglehub`.
2. Merge and structure the folders into a unified `dataset/train` and `dataset/val` structure.
3. Define a PyTorch model (MobileNetV2 or ResNet18) and train it using transfer learning.
4. Save the trained weights (`agrishield_model.pth`) and classes list (`class_names.json`) to be loaded by our FastAPI backend.

## Step 1: Install Dependencies

In [1]:
!pip install kagglehub torch torchvision tqdm albumentations

  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/123.0 MB 1.8 MB/s eta 0:01:08
    ----------

## Step 2: Download Datasets from Kaggle

We use `kagglehub` to download the datasets directly. This doesn't require setting up credentials in Colab!

In [2]:
import kagglehub
import os
import shutil
import json
from glob import glob
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("Downloading Dataset 1: Standard Crops...")
path_crops = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Downloading Dataset 2: Medicinal Plants...")
path_medicinal = kagglehub.dataset_download("jocelyndumlao/medicinal-leaf-health-classification")

print("Downloading Dataset 3: Rust & Powdery Mildew...")
path_rust = kagglehub.dataset_download("rashikrahmanpritom/plant-disease-recognition-dataset")

print("Crops dataset path:", path_crops)
print("Medicinal dataset path:", path_medicinal)
print("Rust dataset path:", path_rust)

g:\New folder (3)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'sklearn'

## Step 3: Merge and Organize Datasets

We will merge all three datasets into a single folder structure:
- `unified_dataset/train/<class_name>/`
- `unified_dataset/val/<class_name>/`

We will map the labels cleanly (e.g., `Tomato___Bacterial_spot`, `Neem___Healthy`, `Generic___Rust`).

In [ ]:
base_dir = "unified_dataset"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

# Clean existing unified dataset first to avoid leftover empty directories (like Generic___Train)
if os.path.exists(base_dir):
    print("Cleaning old directories...")
    shutil.rmtree(base_dir)

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

def copy_images(src_list, dest_folder):
    os.makedirs(dest_folder, exist_ok=True)
    for img_path in src_list:
        if os.path.isfile(img_path):
            shutil.copy(img_path, os.path.join(dest_folder, os.path.basename(img_path)))

# ==========================================
# 1. Processing Samir Bhattarai (Crops) Dataset
# ==========================================
print("Processing Crops Dataset...")
copied_crops = 0
for root, dirs, files in os.walk(path_crops):
    parent_dir = os.path.basename(root)
    grandparent_dir = os.path.basename(os.path.dirname(root))
    
    # PlantVillage classes always have '___' in their names (e.g., Apple___Black_rot)
    if "___" in parent_dir:
        is_train = "train" in grandparent_dir.lower()
        is_val = "val" in grandparent_dir.lower() or "valid" in grandparent_dir.lower()
        
        if is_train or is_val:
            clean_class = parent_dir.replace(" ", "_")
            dest_dir = train_dir if is_train else val_dir
            dest_class_path = os.path.join(dest_dir, clean_class)
            os.makedirs(dest_class_path, exist_ok=True)
            for f in files:
                src_file = os.path.join(root, f)
                if os.path.isfile(src_file) and f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    shutil.copy(src_file, os.path.join(dest_class_path, f))
                    copied_crops += 1
print(f"Crops dataset processed. Copied {copied_crops} images.")

# ==========================================
# 2. Processing Medicinal Leaf Dataset
# ==========================================
print("Processing Medicinal Leaf Dataset...")
# Locate the Background Remove Dataset folder
bg_remove_folders = glob(os.path.join(path_medicinal, "**/Background Remove Dataset"), recursive=True)

if bg_remove_folders:
    med_root = bg_remove_folders[0]
    # Subfolders under med_root are plant types (e.g. Aloe Vera, Neem)
    for plant_folder in os.listdir(med_root):
        plant_path = os.path.join(med_root, plant_folder)
        if os.path.isdir(plant_path):
            # Subfolders under plant type are health states (e.g. Healthy, Diseased, Dried)
            for health_folder in os.listdir(plant_path):
                health_path = os.path.join(plant_path, health_folder)
                if os.path.isdir(health_path):
                    # Formulate unified class name e.g. Medicinal_Neem___Healthy
                    plant_name = plant_folder.replace("Azadirachta Indica (Neem)", "Neem").replace(" ", "_")
                    health_name = health_folder.replace(" ", "_")
                    unified_class = f"Medicinal_{plant_name}___{health_name}"
                    
                    # Get all images in this folder
                    imgs = [os.path.join(health_path, f) for f in os.listdir(health_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                    if len(imgs) > 0:
                        # Perform a train/validation split (80-20)
                        train_imgs, val_imgs = train_test_split(imgs, test_size=0.2, random_state=42)
                        copy_images(train_imgs, os.path.join(train_dir, unified_class))
                        copy_images(val_imgs, os.path.join(val_dir, unified_class))

# ==========================================
# 3. Processing Rust & Powdery Mildew Dataset
# ==========================================
print("Processing Rust & Powdery Mildew Dataset...")
copied_rust = 0
for root, dirs, files in os.walk(path_rust):
    parent_dir = os.path.basename(root)
    grandparent_dir = os.path.basename(os.path.dirname(root))
    
    if parent_dir.lower() in ['healthy', 'powdery', 'rust']:
        is_train = 'train' in grandparent_dir.lower()
        is_val = 'val' in grandparent_dir.lower() or 'valid' in grandparent_dir.lower()
        
        if is_train or is_val:
            unified_class = f"Generic___{parent_dir.capitalize()}"
            dest_dir = train_dir if is_train else val_dir
            dest_class_path = os.path.join(dest_dir, unified_class)
            os.makedirs(dest_class_path, exist_ok=True)
            for f in files:
                src_file = os.path.join(root, f)
                if os.path.isfile(src_file) and f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    shutil.copy(src_file, os.path.join(dest_class_path, f))
                    copied_rust += 1
print(f"Rust & Powdery Mildew dataset processed. Copied {copied_rust} images.")

print("Data merging complete!")
print("Total training classes:", len(os.listdir(train_dir)))
print("Total validation classes:", len(os.listdir(val_dir)))

## Step 4: Define Datasets and DataLoaders

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 64

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_dataset.classes
print("Sample classes:", class_names[:5])
print(f"Total unique classes: {len(class_names)}")

# Save class names to a JSON file
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

## Step 5: Define Deep Learning Model (Transfer Learning)

We will load a pre-trained **MobileNetV2** model. MobileNetV2 is perfect for mobile apps (like Flutter) because it is lightweight, runs fast, and uses minimal memory while maintaining high classification accuracy.

In [ ]:
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

# Load pre-trained MobileNetV2
weights = MobileNet_V2_Weights.DEFAULT
model = mobilenet_v2(weights=weights)

# Freeze the feature extractor layers (optional, but accelerates training)
for param in model.parameters():
    param.requires_grad = False

# Replace classification head
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(class_names))
model = model.to(device)

## Step 6: Training Loop

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
# Only optimize parameters that are not frozen
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)

EPOCHS = 10
best_acc = 0.0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 10)
    
    # Training Phase
    model.train()
    running_loss = 0.0
    running_corrects = 0
    
    for inputs, labels in tqdm(train_loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)
    print(f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validation"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_corrects.double() / len(val_dataset)
    print(f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
    
    # Save best model
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        torch.save(model.state_dict(), "agrishield_model.pth")
        print("Saved new best model!")

print("Training Complete!")
print(f"Best Validation Accuracy: {best_acc:.4f}")

## Step 7: Export Files

Download the following files from your Colab files tab:
1. `agrishield_model.pth` (Put this in the FastAPI directory)
2. `class_names.json` (Put this in the FastAPI directory)